# Sugidanon — Code-Switch Hiligaynon ASR Benchmark

Sugidanon is an open benchmark for testing how speech recognition models handle code-switched Hiligaynon, Tagalog, and English speech.

It shows a simple but important result: current speech models often understand the English and Tagalog words, but struggle with the Hiligaynon itself.

## See it work

Open the notebook and click:

**Runtime → Run all**

That’s it. No local setup needed.

The notebook will:

1. Download the open Sugidanon dataset
2. Validate the benchmark files
3. Score the bundled ASR baseline
4. Print the benchmark results
5. Show the matrix-language erasure gap, or how much the model fails around language-switching points

The bundled baseline runs quickly. Rerunning Whisper live may take a few minutes on the free CPU runtime.

## Links

**Dataset:** https://huggingface.co/datasets/LauelKills/sugidanon-hil-codeswitch
**Code:** https://github.com/Jazztinn/tinig-sa-liwanag
**Live site:** https://tinig-sa-liwanag.vercel.app

## What this proves

Hiligaynon, also known as Ilonggo, is spoken by over 9 million Filipinos, but it is still underrepresented in modern speech technology.

Sugidanon gives developers and researchers a reusable way to measure that gap using open data, per-word language tags, and reproducible scoring.


## 1. Install


In [1]:
!pip install -q openai-whisper huggingface_hub datasets
!apt-get -qq install -y ffmpeg >/dev/null


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 9.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 5.4 MB/s eta 0:00:00


## 2. Get the code and dataset


In [2]:
!git clone -q https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag
from huggingface_hub import snapshot_download
DS = snapshot_download('LauelKills/sugidanon-hil-codeswitch', repo_type='dataset')
print('dataset at:', DS)


/content/tinig-sa-liwanag


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 85 files:   0%|          | 0/85 [00:00<?, ?it/s]

dataset at: /root/.cache/huggingface/hub/datasets--LauelKills--sugidanon-hil-codeswitch/snapshots/ebaea2816e8aa4db1ad0d1a956f2663c2341f48b


## 3. Quick check: validate, score, and package the benchmark
This runs the documented release pipeline on the Hugging Face dataset using the repo's bundled baseline predictions, so it finishes fast. Section 4 then re-runs Whisper live so you can reproduce the same numbers from scratch — that live rerun is the real proof.


In [3]:
!python scripts/build_release.py --annotations $DS/data/annotations --audio $DS/data/audio --predictions data/predictions --output /content/sugidanon_release --overwrite --skip-web


+ python3 scripts/validate.py --kind asr --dir /root/.cache/huggingface/hub/datasets--LauelKills--sugidanon-hil-codeswitch/snapshots/ebaea2816e8aa4db1ad0d1a956f2663c2341f48b/data/annotations
OK   hil_cs_001.json
OK   hil_cs_002.json
OK   hil_cs_003.json
OK   hil_cs_004.json
OK   hil_cs_005.json
OK   hil_cs_006.json
OK   hil_cs_007.json
OK   hil_cs_008.json
OK   hil_cs_009.json
OK   hil_cs_010.json
OK   hil_cs_011.json
OK   hil_cs_012.json
OK   hil_cs_013.json
OK   hil_cs_014.json
OK   hil_cs_015.json
OK   hil_cs_016.json
OK   hil_cs_017.json
OK   hil_cs_018.json
OK   hil_cs_019.json
OK   hil_cs_020.json
OK   hil_cs_021.json
OK   hil_cs_022.json
OK   hil_cs_023.json
OK   hil_cs_024.json
OK   hil_cs_025.json
OK   hil_cs_026.json
OK   hil_cs_027.json
OK   hil_cs_028.json
OK   hil_cs_029.json
OK   hil_cs_030.json
OK   hil_cs_031.json
OK   hil_cs_032.json
OK   hil_cs_033.json
OK   hil_cs_034.json
OK   hil_cs_035.json
OK   hil_cs_036.json
OK   hil_cs_037.json
OK   hil_cs_038.json
OK   hil_cs

## 3b. Reproducibility gate
The benchmark is content-addressed: every annotation and audio file is hashed in `data/benchmark/MANIFEST.json` (version 1.0.0), and the scorer is pinned by hash. This cell fails loudly if anything drifted from the frozen version — that is what makes "the same numbers every time" enforced, not hoped for. See `BENCHMARK.md` for the full protocol.

In [4]:
!python scripts/freeze_benchmark.py --verify

OK: working tree matches frozen benchmark v1.0.1.


## 4. Optional live rerun: Whisper baseline
Whisper has no native Hiligaynon; Tagalog (tl) is the closest, which is the
point. Watch it transcribe the English and Tagalog words well and mangle the
Hiligaynon.


In [ ]:
!python scripts/run_whisper.py --audio-dir $DS/data/audio --out-dir /content/preds --model small --language tl


Loading whisper model: small
100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 86.3MiB/s]
hil_cs_001: Bilangus rin budget natin para sa weekend....
hil_cs_002: Magpakalta sa ngisda kay Malumura pa sa kanina....
hil_cs_003: Nagdiscount ka na iantindahan kaya bumili na lang tayo ng ma...
hil_cs_004: Tamalin na nabukas, dinaw to mong dugang....
hil_cs_005: Planas ang change ang vendor, super violent ang sang bente....
hil_cs_006: Pwede ko mag-drop off sa my corner sa highway....
hil_cs_007: Saka inatayaw kay mali pa ako sa taba'o....


## 5. Score the live Whisper output
This should reproduce the same pattern as the bundled baseline: Hiligaynon
matrix words are harder than the borrowed Tagalog/English switch words.


In [ ]:
!python score.py --ref $DS/data/annotations --hyp /content/preds --ci

## What you just saw

A negative matrix-language erasure gap: words next to a language switch scored better than
the monolingual Hiligaynon. The switch regions carry the borrowed English and
Tagalog words the model already knows; the Hiligaynon matrix is what it cannot
handle. tl-en is near-solved (about 6 percent error); hil-en is the worst
(about 40 percent). The failure scales with how much Hiligaynon is in the
sentence — exactly what this dataset measures.

Swap `--model small` for `large-v3`, or run Meta MMS, to put another model on
the same benchmark. To compare your own ASR system, write one JSON prediction
per clip with `{ "clip_id": "hil_cs_001", "text": "..." }`, then run
`python score.py --ref $DS/data/annotations --hyp your_predictions_dir`.

**Limitations (read before citing numbers).** Preliminary: Whisper *small*, scripted elicited clips, one headline speaker, and a small benchmark size. Per-word language tags for the headline clips are speaker-reviewed. Treat this as a reproducible benchmark scaffold, not a final model ranking.
